In [ ]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
import pandas as pd
import matplotlib.pyplot as plt

import pandas as pd
import plotly.graph_objects as go

In [ ]:
def show_date(dates):
    dates = dates.copy()
    full_range = pd.date_range(start=dates.min(), end=dates.max(), freq='h')
    df = pd.DataFrame({'datetime': full_range})
    df['Value'] = 0
    df.loc[df['datetime'].isin(dates), 'Value'] = 1
    plt.figure(figsize=(12, 6))
    plt.plot(df['datetime'], df['Value'], drawstyle='steps-post', marker='.')
    plt.xlabel('Date and Time')
    plt.ylabel('Value')
    plt.title('Date and Time Presence')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()
    return df[df['Value'] == 0]["datetime"]

In [ ]:
csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df = pd.read_csv(csv_semi_processed_path, encoding='utf-8')
df["datetime"] = pd.to_datetime(df['datetime'])

In [ ]:
df = df[['generation','datetime','name', 'code',"is_good_peak"]]
df.dropna(inplace=True)
name, code = "پرند", "G12"
power_plants = df[['name', 'code']].drop_duplicates()
for _, row in power_plants.iterrows():
    name, code = row['name'], row['code']
    # فیلتر کردن داده‌ها
    df_name_code = df[(df['name'] == name) & (df['code'] == code) & (df["is_good_peak"] >= 3)]
    #df_name_code[df_name_code["generation"] > 200] = None
    #df_name_code[df_name_code["generation"] < 50] = None
    df_name_code.dropna(inplace=True)
    df_name_code.loc[:,"generation"] = df_name_code["generation"].rolling(window=24*7).mean().copy()
    print(df_name_code.head())
    df_name_code.dropna(inplace=True)
    gens = df_name_code['generation'].values
    temp = df_name_code['datetime'].values
    print(df_name_code.head())
    # ایجاد نمودار پراکندگی
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=temp,
        y=gens,
        mode='markers',
        marker=dict(
            size=3,
            opacity=0.6,
            color='blue'
        ),
        name='داده‌های تولید'
    ))

    fig.update_layout(
        title=f'نمودار پراکندگی دمای حس شده vs تولید - {name+"-"+code}',
        xaxis_title='دمای حس شده',
        yaxis_title='تولید (مگاوات)',
        template='plotly_white'
    )
    #fig.show()
    #sinput()
    #print(f"{name}-{code}")
    fig.write_html(f"{project_root}/src/visualization/unit_figs/moving_average/{name}-{code}.html")

In [ ]:
def draw(df_s,row):
    name, code = row['name'], row['code']

    # فیلتر کردن داده‌ها
    df_s2 = df_s[(df_s['name'] == name) & (df_s['code'] == code) & (df_s["is_good_peak"] >= 3)]
    df_merged = pd.merge(df_s2, df_sen, on="datetime", how="inner")
    print(len(df_merged), len(df_s2))

    # انتخاب ستون‌های مورد نیاز و حذف مقادیر NaN
    my_cols = df_merged[['date', 'hour', 'temperature', 'Value', 'generation']].dropna()
    gens = my_cols['generation'].values
    temp_senses = my_cols['Value'].values

    # ایجاد نمودار پراکندگی
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=temp_senses,
        y=gens,
        mode='markers',
        marker=dict(
            size=3,
            opacity=0.6,
            color='blue'
        ),
        name='داده‌های تولید'
    ))

    fig.update_layout(
        title=f'نمودار پراکندگی دمای حس شده vs تولید - {name+"-"+code}',
        xaxis_title='دمای حس شده',
        yaxis_title='تولید (مگاوات)',
        template='plotly_white'
    )

    fig.write_html(f"{project_root}/src/visualization/unit_figs/generation_vs_temeperature/{name}-{code}.html")